---
---
<center>

**<font color='#FF8C00'>.•.•●•.•●⬤●•. Буду рад вашей поддержке! .•●⬤●•.•●•.•.</font>**

<a href="https://www.donationalerts.com/r/politrees" title="Перейти к Donationalerts">
   <img src="https://upload.wikimedia.org/wikipedia/ru/a/ad/DA_Logo_Color.svg" width="200" alt="Donationalerts">
</a>

**Делаю модели на заказ. Подробности в [Telegram](https://t.me/Politrees2)**

---

**Будьте в курсе всех обновлений и новостей! Подписывайтесь на мой [Telegram-канал](https://t.me/politrees)**

</center>

---

# **MuXVS**

In [ ]:
# @title <big> ⬇️ **Установка**

import os

import torch
from google.colab import drive
from IPython.display import clear_output
from ipywidgets import Button

version = "arena/01a0c5ce-muxvs"  # @param ["arena/01a0c5ce-muxvs", "main"] {"allow-input":true}
git = "https://github.com/MuXolotl/MuXVS"
dir = "/co" + "nte" + "nt/Mu" + "XVS"

print("Проверка доступности GPU...")
if not torch.cuda.is_available():
    raise Exception(
        "\033[91mGPU недоступен!\033[0m\nК сожалению, у вас нет доступа к GPU на вашем текущем аккаунте. Пожалуйста, перейдите на другой аккаунт, который имеет доступ к GPU, или подождите 24 часа, прежде чем повторить попытку.",
    )
print("\033[92mGPU доступен!\033[0m")

if not os.path.isdir("/content/drive"):
    print("\nПодключение к Google Drive...")
    drive.mount("/content/drive")
if not os.path.exists("/content/dataset"):
    os.makedirs("/content/dataset")

!git clone --depth 1 $git $dir --branch $version --single-branch &> /dev/null
%cd $dir

!pip install --no-cache-dir -q uv
!uv pip install --no-cache-dir -q -r requirements.txt

!rm -rf /content/sample_data/

clear_output()
Button(description="✔ Готово!", button_style="success")

In [ ]:
# @title <big> ⬇️ **Запуск интерфейса**

# @markdown ### **Способ запуска**:
launch_method = "Gradio (share)"  # @param ["Gradio (share)", "localtunnel", "cloudflared", "localhost.run", "Pinggy", "ngrok"]
# @markdown Работают стабильно без VPN:
# @markdown - localtunnel (ссылка одна на весь запуск)
# @markdown - localhost.run (при обрыве ссылка МЕНЯЕТСЯ — берите новую из вывода!)

# @markdown ---

# @markdown ### **Токен ngrok**: <small><small>*(можно получить на [ngrok.com](ngrok.com) → Your Authtoken)</small></small>*
ngrok_token = ""  # @param {type:"string"}

import os
import re
import socket
import subprocess
import sys
import time
import uuid

import ipywidgets as widgets
from IPython.display import display

PROJECT_DIR = "/content/MuXVS"
PORT = 4000
STARTUP_TIMEOUT = 180

if not os.path.exists(PROJECT_DIR):
    raise FileNotFoundError(f"Директория {PROJECT_DIR} не найдена. Сначала запустите ячейку установки.")

os.chdir(PROJECT_DIR)


def print_url(url):
    print("🔗 Ссылка для доступа в интерфейс:", url)


def wait_for_server(port, timeout=STARTUP_TIMEOUT):
    start = time.time()
    while time.time() - start < timeout:
        try:
            with socket.create_connection(("127.0.0.1", port), timeout=2):
                return True
        except (ConnectionRefusedError, OSError):
            time.sleep(2)
    raise TimeoutError(f"Сервер не запустился на порту {port} за {timeout} секунд.")


def start_app(port):
    subprocess.Popen(
        [sys.executable, "app.py", "--port", str(port), "--no-share"],
        stdout=open("app.log", "w"),
        stderr=subprocess.STDOUT,
    )
    print(f"⏳ Ожидание запуска сервера на порту {port}...")
    wait_for_server(port)
    print(f"✅ Сервер запущен на порту {port}\n")


def ensure_ssh_key():
    key_path = os.path.expanduser("~/.ssh/id_rsa")
    if not os.path.exists(key_path):
        os.makedirs(os.path.dirname(key_path), exist_ok=True)
        subprocess.run(["ssh-keygen", "-t", "rsa", "-N", "", "-f", key_path, "-q"], check=True)


def ssh_tunnel_loop(ssh_args, url_patterns, reconnect_delay=5):
    url_res = [re.compile(p) for p in url_patterns]

    connections = 0
    while True:
        proc = subprocess.Popen(
            ssh_args,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        link_printed = False
        try:
            for line in proc.stdout:
                line = line.strip()
                if not line or line.startswith("RB:"):
                    continue

                if not link_printed:
                    for pattern in url_res:
                        m = pattern.search(line)
                        if m:
                            connections += 1
                            if connections > 1:
                                print()
                                print("=" * 60)
                                print("⚠️ СОЕДИНЕНИЕ ОБНОВИЛОСЬ — старая вкладка МЁРТВА!")
                                print("   Откройте интерфейс по НОВОЙ ссылке:")
                                print("=" * 60)
                            print_url(m.group(0).rstrip(").,"))
                            link_printed = True
                            break
        finally:
            proc.wait()

        print(f"\n⚠️  Соединение потеряно, переподключение через {reconnect_delay} сек...\n")
        time.sleep(reconnect_delay)


def keep_alive():
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        pass


# ======== Gradio (share) ========
if launch_method == "Gradio (share)":
    print("⚠️  Туннель идёт через Cloudflare — у некоторых может не работать.")
    print("   Если не грузится — попробуйте localhost.run или localtunnel\n")
    !python app.py


# ======== localtunnel ========
elif launch_method == "localtunnel":
    subprocess.run(["npm", "install", "-g", "localtunnel"], capture_output=True)
    LT_SUBDOMAIN = globals().get("LT_SUBDOMAIN") or f"muxvs-{uuid.uuid4().hex[:8]}"
    start_app(PORT)

    ip_address = subprocess.check_output(["curl", "-s", "ifconfig.me"]).decode().strip()

    print("=" * 50)
    print("Пароль (IP-адрес) для доступа к ссылке ниже:")
    display(widgets.Text(value=ip_address, description="Пароль (IP):", disabled=True))
    print("=" * 50 + "\n")

    !lt --port {PORT} --subdomain {LT_SUBDOMAIN}


# ======== cloudflared ========
elif launch_method == "cloudflared":
    print("⚠️  Туннель идёт через Cloudflare — у некоторых пользователей может не работать.")
    print("   Если не грузится — попробуйте localhost.run или localtunnel\n")

    subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)

    if os.path.exists("cloudflared"):
        os.remove("cloudflared")

    subprocess.run(
        [
            "wget",
            "-q",
            "-c",
            "-N",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O",
            "cloudflared",
        ],
        check=True,
    )
    os.chmod("cloudflared", 0o755)

    start_app(PORT)

    subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
        stdout=open("cloudflared.log", "w"),
        stderr=subprocess.STDOUT,
    )

    url_re = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
    public_url = None

    for _ in range(30):
        time.sleep(2)
        if os.path.exists("cloudflared.log"):
            with open("cloudflared.log") as f:
                m = url_re.search(f.read())
                if m:
                    public_url = m.group(0)
                    break

    if public_url:
        print_url(public_url)
    else:
        print("❌ Не удалось получить ссылку. Лог:")
        !tail -n 80 cloudflared.log

    keep_alive()


# ======== localhost.run ========
elif launch_method == "localhost.run":
    start_app(PORT)
    ensure_ssh_key()

    ssh_tunnel_loop(
        [
            "ssh",
            "-tt",
            "-o",
            "StrictHostKeyChecking=no",
            "-o",
            "UserKnownHostsFile=/dev/null",
            "-o",
            "ServerAliveInterval=30",
            "-o",
            "ServerAliveCountMax=3",
            "-o",
            "ConnectTimeout=15",
            "-R",
            f"80:127.0.0.1:{PORT}",
            "nokey@localhost.run",
        ],
        url_patterns=[
            r"https://[a-z0-9]+\.lhr\.life",
        ],
    )


# ======== Pinggy ========
elif launch_method == "Pinggy":
    start_app(PORT)
    ensure_ssh_key()

    ssh_tunnel_loop(
        [
            "ssh",
            "-T",
            "-o",
            "StrictHostKeyChecking=no",
            "-o",
            "UserKnownHostsFile=/dev/null",
            "-o",
            "ServerAliveInterval=30",
            "-o",
            "ServerAliveCountMax=3",
            "-o",
            "ConnectTimeout=15",
            "-p",
            "443",
            f"-R0:127.0.0.1:{PORT}",
            "http@a.pinggy.io",
        ],
        url_patterns=[
            r"https://[a-z0-9-]+\.a\.free\.pinggy\.link",
            r"https://[a-z0-9-]+\.free\.pinggy\.link",
            r"https://[a-z0-9-]+\.a\.pinggy\.link",
            r"https://[a-z0-9-]+\.pinggy\.link",
        ],
    )


# ======== ngrok ========
elif launch_method == "ngrok":
    if not ngrok_token:
        raise ValueError("Для ngrok нужен токен: ngrok.com → Your Authtoken")

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], capture_output=True)
    from pyngrok import ngrok as ngrok_module

    ngrok_module.set_auth_token(ngrok_token)
    start_app(PORT)

    public_url = ngrok_module.connect(PORT, bind_tls=True)
    print_url(public_url)

    try:
        keep_alive()
    finally:
        ngrok_module.disconnect(public_url)

In [ ]:
# @title <big>📊 **TensorBoard**
# @markdown Запустите во время обучения — графики появятся прямо в ячейке.
SAVE_DIR = "/content/drive/MyDrive/MuXVS"  # @param {type:"string"}

%load_ext tensorboard
%tensorboard --logdir {SAVE_DIR}

## <big>
---
    Все файлы, созданные в процессе тренировки,
    автоматически сохраняются на Google Диск в папку MuXVS.

    * Путь к .pth файлу:
       MuXVS / [Имя Модели] / [Имя Модели]_e10_s500_last.pth        - Финальная модель
       MuXVS / [Имя Модели] / weights / [Имя Модели]_e10_s500.pth   - Промежуточные модели
    * Путь к .index файлу:
       MuXVS / [Имя Модели] / [Имя Модели].index                    - Индексный файл

    Также, если вы включите параметр "save_to_zip",
    то по окончании тренировки будет создан ZIP-архив,
    в котором будут содержаться: финальная модель и индексный файл.

    * Путь к .zip файлу:
       MuXVS / [Имя Модели] / [Имя Модели].zip                      - ZIP-архив
---

<center>

**MuXVS — конвертация голоса, TTS, разделение аудио и обучение в одном интерфейсе.**

**[PolTrain](https://github.com/Politrees/PolTrain) — обучение RVC-моделей.**

**[PolGen](https://github.com/Politrees/PolGen) — преобразование голоса и текста в речь.**

---

### Количество посещений:

<img src="https://counter.seku.su/cmoe?name=MuXVS&theme=mbs" /><br>